# Plan-and-Execute

Planner/executor separation with typed plan steps, `Command`-based orchestration, and execution tracing.

## Walkthrough

1. Configure stub or live mode.
2. Optionally set different planner and executor models for live runs.
3. Run the workflow and inspect the structured plan plus execution trace.

- Default notebook mode: `stub`
- Live mode requires `OPENAI_API_KEY` in the notebook kernel environment before switching `MODE` to `live`.
- `MAX_STEPS` caps planner output to keep latency bounded.

In [ ]:
MODE = "stub"
MODEL = "gpt-4.1-mini"
PLANNER_MODEL = MODEL
EXECUTOR_MODEL = MODEL
MAX_STEPS = 4
TASK = "Plan and execute a short answer explaining when planner/executor separation helps."

## Notes on live mode

- Keep `MODE = "stub"` for deterministic offline execution.
- Switch to `MODE = "live"` only after setting `OPENAI_API_KEY`.
- In Jupyter, `%env OPENAI_API_KEY=sk-...` is the quickest way to set the key for the current kernel.
- In Colab, run `import os; os.environ["OPENAI_API_KEY"] = "sk-..."` before executing the workflow cell.
- Override `PLANNER_MODEL` or `EXECUTOR_MODEL` when you want to compare a cheaper planner against a stronger executor (or vice versa).

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src").exists():
    ROOT = ROOT.parent
for path in (ROOT, ROOT / "src"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from examples.planning_and_execute_example import PlanStep, run_demo

## Run the workflow

The planner emits structured `PlanStep` entries, and the executor processes each step in order while the orchestrator appends an execution trace.

In [ ]:
RESULT = run_demo(
    TASK,
    mode=MODE,
    model=MODEL,
    planner_model=PLANNER_MODEL,
    executor_model=EXECUTOR_MODEL,
    max_steps=MAX_STEPS,
)
RESULT

## Inspect the structured plan and trace

These cells make the planner/executor split explicit in notebook form instead of only showing the final dictionary.

In [ ]:
PLAN_STEPS = [PlanStep.model_validate(step) for step in RESULT["plan_steps"]]
[(step.step_id, step.objective) for step in PLAN_STEPS]

In [ ]:
RESULT["execution_trace"]

In [ ]:
RESULT["final_output"]